# 🧪 O-ISAC Extraction Benchmark Lab

Bu notebook, **Pipeline V4 (CoT + Vision)** performansını mevcut (Legacy) extraction sonuçlarıyla kıyaslamak için tasarlanmıştır.

### 🎯 Hedef
Görsel analiz verilerinin (`visual_analysis.txt`) extraction kalitesine, veri doğruluğuna ve akıl yürütme derinliğine olan etkisini kantitatif ve kalitatif olarak ölçmek.

---

In [16]:
# @title 0. Bağımlılıkları Yükle
!pip install groq -q
print("✅ Gerekli paketler yüklendi.")

✅ Gerekli paketler yüklendi.


In [17]:
# @title 1. Kurulum ve Bağlantılar
from google.colab import drive, userdata
import os, sys, json, glob
import pandas as pd
from IPython.display import display, HTML

drive.mount('/content/drive')
PROJECT_ROOT = '/content/drive/MyDrive/AKU_WorkSpace/survey_fdgit/OISAC_PRISMA_COMST'
sys.path.append(os.path.join(PROJECT_ROOT, 'analysis/nb'))

# API Anahtarlarını Ayarla (Colab Secrets'tan çeker)
try:
    os.environ["GROQ_API_KEY"] = userdata.get('GROQ_API_KEY')
    print("🔑 API Anahtarları yüklendi.")
except:
    print("⚠️ GROQ_API_KEY Colab Secrets içinde bulunamadı. Lütfen ekleyin!")

import extraction_pipeline_v3 as v3
import extraction_pipeline_v4 as v4
from extraction_pipeline_v4 import ConfigV4

print("✅ Pipeline motorları yüklendi.")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
🔑 API Anahtarları yüklendi.
✅ Pipeline motorları yüklendi.


## 📂 2. Kıyaslama Örneği Seçimi

`data/test_out` klasöründe bulunan mevcut (Legacy) sonuçlardan birini seçin.

In [22]:
# @title Mevcut Legacy Dosyalarını Listele
legacy_dir = os.path.join(PROJECT_ROOT, "data/test_out")
legacy_files = [f for f in os.listdir(legacy_dir) if f.startswith("O_ISAC_") and f.endswith(".json")]
legacy_ids = sorted(list(set([f.split('_simulation')[0].split('.json')[0] for f in legacy_files])))

print(f"📂 {len(legacy_ids)} adet legacy çalışma bulundu.")
print("Örnek ID'ler:", legacy_ids[:10], "...")

PAPER_ID = "O_ISAC_057" # @param {type:"string"}

# 1. Mevcut (Legacy) Sonucu Yükle
legacy_matches = glob.glob(os.path.join(legacy_dir, f"{PAPER_ID}*.json"))
if not legacy_matches:
    print(f"❌ Legacy data NOT found for {PAPER_ID} in {legacy_dir}")
    legacy_data = {}
else:
    with open(legacy_matches[0], 'r', encoding='utf-8') as f:
        legacy_data = json.load(f)
    print(f"📦 Legacy data loaded: {os.path.basename(legacy_matches[0])}")

📂 66 adet legacy çalışma bulundu.
Örnek ID'ler: ['O_ISAC_001', 'O_ISAC_002', 'O_ISAC_003', 'O_ISAC_004', 'O_ISAC_005', 'O_ISAC_006', 'O_ISAC_007', 'O_ISAC_008', 'O_ISAC_009', 'O_ISAC_010'] ...
📦 Legacy data loaded: O_ISAC_057_experiment.json


## 👁️ 3. V4 Extraction Yükle

Seçilen makale için V4 sonuçlarını yüklüyoruz.

In [23]:
FORCE_RUN = False # @param {type:"boolean"}

v4_result_path = os.path.join(PROJECT_ROOT, "data/ext_res_v4", f"{PAPER_ID}_v4.json")

if FORCE_RUN or not os.path.exists(v4_result_path):
    print(f"🧠 {PAPER_ID} için V4 (Vision + CoT) çalıştırılıyor...")
    ConfigV4.init_dirs()
    # Not: checkpoint dosyası isterseniz buraya tanımlayabilirsiniz.
    # results = v4.phase3_integrated_extraction(limit=None)
    # Biz burada benchmark için tek makale üzerinden gidebiliriz veya toplu sonuçtan çekebiliriz.
    pass

if os.path.exists(v4_result_path):
    with open(v4_result_path, 'r', encoding='utf-8') as f:
        v4_data = json.load(f)
    print(f"✅ V4 Data Loaded: {os.path.basename(v4_result_path)}")
else:
    print(f"⚠️ V4 Sonucu bulunamadı: {v4_result_path}")
    v4_data = {}

✅ V4 Data Loaded: O_ISAC_057_v4.json


## 📊 4. Yan Yana Kıyaslama (Side-by-Side)

Şimdi iki versiyon arasındaki temel farkları tablo olarak görüyoruz.

In [24]:
# @title Veri Karşılaştırma Fonksiyonu
def compare_extractions(legacy, v4):
    rows = []

    # Karşılaştırılacak anahtar alanlar
    comp_keys = {
        "Scenario": ["scenario_overview.scenario_context", "scenario_level_details[0].scenario_context"],
        "Data Rate": ["scenario_level_details[0].metrics.data_rate", "scenario_level_details[0].metrics.data_rate"],
        "Resolution": ["scenario_level_details[0].metrics.localization_resolution", "scenario_level_details[0].metrics.sensing_resolution"],
        "Visual Insight": ["None", "reasoning_trace[0].value"], # V4'te reasoning var
        "Quality Score": ["quality_assessment.overall_confidence", "quality_assessment.overall_confidence"]
    }

    def get_val(data, path):
        if path == "None" or not data: return "NR"
        try:
            curr = data
            for p in path.replace('[', '.').replace(']', '').split('.'):
                if p.isdigit():
                    curr = curr[int(p)]
                else:
                    curr = curr.get(p, "NR")
            return curr
        except:
            return "NR"

    # Bibliyografik Bilgiler
    rows.append({"Field": "📄 Paper Title", "Legacy": get_val(legacy, "bibliographic_info.title"), "V4 (CoT + Vision)": get_val(v4, "bib_info.title")})

    # Senaryo ve Metrikler
    l_sc = get_val(legacy, "scenario_level_details.0.scenario_context")
    v_sc = get_val(v4, "scenario_level_details.0.scenario_context")
    rows.append({"Field": "🏢 Scenario Context", "Legacy": l_sc, "V4 (CoT + Vision)": v_sc})

    l_dr = get_val(legacy, "scenario_level_details.0.metrics.data_rate")
    v_dr = get_val(v4, "scenario_level_details.0.metrics.data_rate")
    rows.append({"Field": "🚀 Data Rate", "Legacy": l_dr, "V4 (CoT + Vision)": v_dr})

    l_res = get_val(legacy, "scenario_level_details.0.metrics.localization_resolution")
    v_res = get_val(v4, "scenario_level_details.0.metrics.sensing_resolution")
    rows.append({"Field": "🎯 Resolution", "Legacy": l_res, "V4 (CoT + Vision)": v_res})

    # Akıl Yürütme ve Görsel Analiz (V4'e özel)
    v_reason = get_val(v4, "reasoning_trace.0.value")
    rows.append({"Field": "🧠 Reasoning/Visual", "Legacy": "❌ (No Vision Pipeline)", "V4 (CoT + Vision)": v_reason[:300] + "..." if v_reason != "NR" else "NR"})

    df = pd.DataFrame(rows)

    # Styling: V4'ün Legacy'den daha iyi olduğu durumları vurgulayalım (Eskiden NR olup şimdi dolu olanlar)
    def highlight_improvements(s):
        is_impr = (s['Legacy'] == "NR" or "❌" in str(s['Legacy'])) and (s['V4 (CoT + Vision)'] != "NR" and s['V4 (CoT + Vision)'] != "")
        return ['background-color: #d4edda' if is_impr else '' for _ in s]

    return df.style.apply(highlight_improvements, axis=1)

if legacy_data or v4_data:
    display(compare_extractions(legacy_data, v4_data))
else:
    print("❌ Kıyaslanacak veri bulunamadı.")

,Field,Legacy,V4 (CoT + Vision)
0,📄 Paper Title,NR,NR
1,🏢 Scenario Context,NR,NR
2,🚀 Data Rate,NR,NR
3,🎯 Resolution,NR,NR
4,🧠 Reasoning/Visual,❌ (No Vision Pipeline),"The paper includes several figures, including Figure 1, which shows the schematic diagram of the proposed 60-GHz mm-wave joint radar-communication system, and Figure 2, which shows the optical spectrum and instantaneous frequency of the mm-wave LFM-OOK joint signal...."
